In [3]:
"""
MNIST 手写数字分类：SGD vs NG 对比
=================================================
依赖：numpy（仅需标准库 + numpy）

运行：python3 mnist_sgd_vs_ng.py

网络结构：784 → 128 (ReLU) → 10 (Softmax)
优化器：
  - SGD：θ ← θ − η∇L
  - 自然梯度（KFAC）：θ ← θ − η·F⁻¹∇L，Fisher 矩阵用 Kronecker 分解近似
"""

import numpy as np
import json
import time
import urllib.request
import gzip
import struct
import os

# ──────────────────────────────────────────────────────────
# 1. MNIST 数据加载
# ──────────────────────────────────────────────────────────
MNIST_URLS = {
    "train_images": "https://ossci-datasets.s3.amazonaws.com/mnist/train-images-idx3-ubyte.gz",
    "train_labels": "https://ossci-datasets.s3.amazonaws.com/mnist/train-labels-idx1-ubyte.gz",
    "test_images":  "https://ossci-datasets.s3.amazonaws.com/mnist/t10k-images-idx3-ubyte.gz",
    "test_labels":  "https://ossci-datasets.s3.amazonaws.com/mnist/t10k-labels-idx1-ubyte.gz",
}

MNIST_URLS_FALLBACK = {
    "train_images": "https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz",
    "train_labels": "https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz",
    "test_images":  "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-images-idx3-ubyte.gz",
    "test_labels":  "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz",
}

CACHE_DIR = os.path.join(os.path.expanduser("~"), "mnist_cache")
os.makedirs(CACHE_DIR, exist_ok=True)


def download_mnist():
    """下载 MNIST 数据集（如已缓存则跳过，主源失败自动切换备用源）"""
    paths = {}
    for name in MNIST_URLS:
        fname = os.path.join(CACHE_DIR, name + ".gz")
        if not os.path.exists(fname):
            for url in (MNIST_URLS[name], MNIST_URLS_FALLBACK[name]):
                try:
                    print(f"  下载 {name} ...")
                    urllib.request.urlretrieve(url, fname)
                    break
                except Exception as e:
                    print(f"    失败 ({e})，尝试备用源...")
                    if os.path.exists(fname):
                        os.remove(fname)
            else:
                raise RuntimeError(f"无法下载 {name}，请手动放置文件到 {CACHE_DIR}")
        paths[name] = fname
    return paths


def load_images(path):
    """加载图像并归一化到 [0, 1]"""
    with gzip.open(path, "rb") as f:
        _, n, rows, cols = struct.unpack(">IIII", f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n, rows * cols).astype(np.float32) / 255.0


def load_labels(path):
    """加载标签"""
    with gzip.open(path, "rb") as f:
        struct.unpack(">II", f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.astype(np.int32)


def one_hot(y, K=10):
    """转换为 one-hot 编码"""
    oh = np.zeros((len(y), K), dtype=np.float32)
    oh[np.arange(len(y)), y] = 1.0
    return oh


# ──────────────────────────────────────────────────────────
# 2. 两层 MLP 神经网络
#    结构：784 → 128（ReLU）→ 10（Softmax）
# ──────────────────────────────────────────────────────────
class MLP:
    """
    两层全连接神经网络。

    参数
    ----
    seed : int
        随机种子（保证 SGD 和 NG 从相同初始点出发）
    """

    def __init__(self, seed: int = 42):
        rng = np.random.default_rng(seed)
        # He 初始化
        self.W1 = (rng.standard_normal((784, 128)) * np.sqrt(2 / 784)).astype(np.float32)
        self.b1 = np.zeros(128, dtype=np.float32)
        self.W2 = (rng.standard_normal((128, 10)) * np.sqrt(2 / 128)).astype(np.float32)
        self.b2 = np.zeros(10, dtype=np.float32)
        self._cache: dict = {}

    # ── 前向传播 ──────────────────────────────────────────
    def forward(self, X: np.ndarray) -> np.ndarray:
        """
        前向传播。

        Returns
        -------
        p : (B, 10) softmax 输出概率
        """
        z1 = X @ self.W1 + self.b1          # (B, 128)
        a1 = np.maximum(0, z1)              # ReLU
        z2 = a1 @ self.W2 + self.b2         # (B, 10)
        # 数值稳定 softmax
        ez = np.exp(z2 - z2.max(axis=1, keepdims=True))
        p  = ez / ez.sum(axis=1, keepdims=True)
        self._cache = {"X": X, "z1": z1, "a1": a1, "p": p}
        return p

    # ── 损失（交叉熵）─────────────────────────────────────
    def loss(self, p: np.ndarray, y_oh: np.ndarray) -> float:
        return float(-np.mean(np.sum(y_oh * np.log(p + 1e-9), axis=1)))

    # ── 准确率 ────────────────────────────────────────────
    def accuracy(self, X: np.ndarray, y: np.ndarray) -> float:
        return float(np.mean(self.forward(X).argmax(axis=1) == y))

    # ── 反向传播（普通梯度）──────────────────────────────
    def gradients(self, y_oh: np.ndarray):
        """
        计算参数梯度（反向传播）。

        Returns
        -------
        grads : tuple (gW1, gb1, gW2, gb2)
        deltas: dict  {'d1': ..., 'd2': ...}  用于 KFAC 计算 G 矩阵
        """
        X, z1, a1, p = (self._cache[k] for k in ("X", "z1", "a1", "p"))
        B = X.shape[0]

        d2 = (p - y_oh) / B                    # (B, 10)  输出层误差
        gW2 = a1.T @ d2                         # (128, 10)
        gb2 = d2.sum(axis=0)                    # (10,)

        d1 = (d2 @ self.W2.T) * (z1 > 0)       # (B, 128)  隐层误差（ReLU 导数）
        gW1 = X.T @ d1                          # (784, 128)
        gb1 = d1.sum(axis=0)                    # (128,)

        return (gW1, gb1, gW2, gb2), {"d1": d1, "d2": d2}

    # ── 参数更新 ──────────────────────────────────────────
    def apply_update(self, updates: tuple) -> None:
        self.W1 -= updates[0]
        self.b1 -= updates[1]
        self.W2 -= updates[2]
        self.b2 -= updates[3]


# ──────────────────────────────────────────────────────────
# 3. 自然梯度优化器
#    核心思想：用 F⁻¹ 校正梯度方向
#    F（Fisher 信息矩阵）≈ Aˡ ⊗ Gˡ（Kronecker 分解）
#    其中 Aˡ = E[aˡ⁻¹(aˡ⁻¹)ᵀ]，Gˡ = E[δˡ(δˡ)ᵀ]
# ──────────────────────────────────────────────────────────
class KFACOptimizer:
    """
    Kronecker-Factored Approximate Curvature (KFAC) 优化器。

    参数
    ----
    lr       : 学习率
    damping  : Tikhonov 正则化系数（避免 Fisher 矩阵奇异）
    freq     : Kronecker 因子更新频率（每 freq 个 mini-batch 更新一次）
    decay    : 指数移动平均衰减系数
    """

    def __init__(self, lr: float = 0.01, damping: float = 1e-3,
                 freq: int = 1, decay: float = 0.9):
        self.lr = lr
        self.lam = damping
        self.freq = freq
        self.decay = decay
        self.step = 0
        # Kronecker 因子（每层两个：A 和 G）
        self.A: list = [None, None]   # 输入激活协方差
        self.G: list = [None, None]   # 输出梯度协方差

    def _ema(self, old, new):
        """指数移动平均"""
        return new if old is None else self.decay * old + (1 - self.decay) * new

    def compute_natural_grad(self, model: MLP, y_oh: np.ndarray,
                              grads: tuple, deltas: dict) -> tuple:
        """
        计算自然梯度更新量。

        步骤：
        1. 估算 Kronecker 因子 A（输入协方差）和 G（梯度协方差）
        2. 用 Tikhonov 正则化后求逆
        3. 自然梯度 = G⁻¹ · ∇W · A⁻¹

        Parameters
        ----------
        model  : MLP 实例（包含缓存的前向激活值）
        y_oh   : (B, 10) one-hot 标签
        grads  : (gW1, gb1, gW2, gb2) 普通梯度
        deltas : {'d1', 'd2'} 各层误差信号

        Returns
        -------
        natural grads : tuple (ngW1, gb1, ngW2, gb2)
        """
        c = model._cache
        X, a1 = c["X"], c["a1"]
        d1, d2 = deltas["d1"], deltas["d2"]
        B = X.shape[0]

        # ── 更新 Kronecker 因子（EMA 保证稳定性）──────────
        if self.step % self.freq == 0:
            # 层 1：A1 = E[x·xᵀ]，G1 = E[δ1·δ1ᵀ]
            self.A[0] = self._ema(self.A[0], (X.T @ X) / B)
            self.G[0] = self._ema(self.G[0], (d1.T @ d1) * B)
            # 层 2：A2 = E[a1·a1ᵀ]，G2 = E[δ2·δ2ᵀ]
            self.A[1] = self._ema(self.A[1], (a1.T @ a1) / B)
            self.G[1] = self._ema(self.G[1], (d2.T @ d2) * B)
        self.step += 1

        gW1, gb1, gW2, gb2 = grads
        lam = self.lam

        def natural(A, G, gW):
            """
            自然梯度：ng = (G + λI)⁻¹ · gW.T · (A + λI)⁻¹，再转置回 gW 的形状。

            gW  : (in_dim, out_dim)   e.g. (784, 128) 或 (128, 10)
            A   : (in_dim,  in_dim)   输入激活协方差
            G   : (out_dim, out_dim)  梯度信号协方差

            Kronecker 自然梯度公式（Martens 2014）：
              vec(ng) = (A ⊗ G)⁻¹ vec(gW)
            矩阵形式等价于：
              ng.T = G⁻¹ · gW.T · A⁻¹
              ng   = (G⁻¹ · gW.T · A⁻¹).T        ← 形状恢复为 (in_dim, out_dim)
            """
            Ar_inv = np.linalg.inv(A + lam * np.eye(A.shape[0], dtype=np.float32))
            Gr_inv = np.linalg.inv(G + lam * np.eye(G.shape[0], dtype=np.float32))
            try:
                # (out×out) @ (out×in) @ (in×in) = (out×in)，转置得 (in×out)
                return (Gr_inv @ gW.T @ Ar_inv).T
            except np.linalg.LinAlgError:
                return gW

        ngW1 = natural(self.A[0], self.G[0], gW1)
        ngW2 = natural(self.A[1], self.G[1], gW2)
        return (ngW1, gb1, ngW2, gb2)


# ──────────────────────────────────────────────────────────
# 4. 通用训练函数
# ──────────────────────────────────────────────────────────
def train(
    method: str,           # "sgd" 或 "ng"
    X_train, y_train,
    X_test,  y_test,
    epochs: int   = 30,
    batch_size: int = 128,
    lr: float     = None,
    seed: int     = 42,
) -> dict:
    """
    训练并返回历史记录。

    SGD  默认学习率：0.08
    NG   默认学习率：0.01（Fisher 矩阵已做尺度调整，无需大学习率）
    """
    lr = lr or (0.08 if method == "sgd" else 0.01)
    model = MLP(seed=seed)
    if method == "ng":
        opt = KFACOptimizer(lr=lr, damping=3e-3, freq=1, decay=0.9)

    rng = np.random.default_rng(seed)
    n = X_train.shape[0]
    hist = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": [], "time": []}
    t0 = time.time()

    label = "SGD" if method == "sgd" else "自然梯度 NG-KFAC"
    print(f"\n{'─'*56}")
    print(f"  方法: {label}   lr={lr}   batch={batch_size}   epochs={epochs}")
    print(f"{'─'*56}")

    for ep in range(1, epochs + 1):
        idx = rng.permutation(n)
        ep_loss, nb = 0.0, 0

        for s in range(0, n, batch_size):
            bi  = idx[s:s + batch_size]
            Xb  = X_train[bi]
            yb  = one_hot(y_train[bi])

            p   = model.forward(Xb)
            ep_loss += model.loss(p, yb)
            nb  += 1

            grads, deltas = model.gradients(yb)

            if method == "sgd":
                # ── SGD：直接用梯度更新 ──────────────────
                updates = tuple(lr * g for g in grads)
            else:
                # ── NG：用自然梯度更新 ───────────────────
                ng = opt.compute_natural_grad(model, yb, grads, deltas)
                updates = tuple(lr * g for g in ng)

            model.apply_update(updates)

        # ── Epoch 评估 ───────────────────────────────────
        tr_acc  = model.accuracy(X_train[:5000], y_train[:5000])
        te_acc  = model.accuracy(X_test, y_test)
        p_test  = model.forward(X_test)
        te_loss = model.loss(p_test, one_hot(y_test))
        elapsed = round(time.time() - t0, 2)

        hist["train_loss"].append(float(ep_loss / nb))
        hist["test_loss"].append(float(te_loss))
        hist["train_acc"].append(float(tr_acc))
        hist["test_acc"].append(float(te_acc))
        hist["time"].append(elapsed)

        print(f"  Epoch {ep:2d}/{epochs} | "
              f"TrainLoss={ep_loss/nb:.4f} | "
              f"TestLoss={te_loss:.4f} | "
              f"TrainAcc={tr_acc*100:.2f}% | "
              f"TestAcc={te_acc*100:.2f}% | "
              f"{elapsed:.1f}s")

    print(f"\n  ✓ 最终测试准确率: {hist['test_acc'][-1]*100:.2f}%")
    return hist


# ──────────────────────────────────────────────────────────
# 5. 主程序
# ──────────────────────────────────────────────────────────
def main():
    print("=" * 56)
    print("  MNIST 手写数字分类：SGD vs NG")
    print("=" * 56)

    # ── 加载数据 ────────────────────────────────────────
    print("\n[1/3] 下载 & 加载 MNIST 数据集...")
    paths   = download_mnist()
    X_train = load_images(paths["train_images"])
    y_train = load_labels(paths["train_labels"])
    X_test  = load_images(paths["test_images"])
    y_test  = load_labels(paths["test_labels"])
    print(f"  训练集: {X_train.shape}  测试集: {X_test.shape}")

    # ── 训练 ─────────────────────────────────────────────
    print("\n[2/3] 训练两种优化器...")
    shared = dict(X_train=X_train, y_train=y_train,
                  X_test=X_test, y_test=y_test,
                  epochs=30, batch_size=128, seed=42)

    h_sgd = train(method="sgd", lr=0.08,  **shared)
    h_ng  = train(method="ng",  lr=0.01,  **shared)

    # ── 保存结果 ─────────────────────────────────────────
    print("\n[3/3] 保存结果...")
    results = {
        "sgd": {k: [float(v) for v in vs] for k, vs in h_sgd.items()},
        "ng":  {k: [float(v) for v in vs] for k, vs in h_ng.items()},
    }
    result_path = os.path.join(os.path.expanduser("~"), "mnist_results.json")
    with open(result_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  结果已保存至 {result_path}")

    # ── 对比摘要 ─────────────────────────────────────────
    print("\n" + "=" * 56)
    print("  对比摘要")
    print("=" * 56)
    hdr = f"  {'指标':<20} {'SGD':>10} {'NG-KFAC':>12}"
    print(hdr)
    print(f"  {'─'*43}")

    def row(name, sgd_val, ng_val, fmt=".4f"):
        print(f"  {name:<20} {sgd_val:>{10}{fmt}} {ng_val:>{12}{fmt}}")

    row("最终测试损失",      h_sgd["test_loss"][-1], h_ng["test_loss"][-1])
    row("最终测试准确率",    h_sgd["test_acc"][-1],  h_ng["test_acc"][-1])
    bsg = h_sgd["test_acc"].index(max(h_sgd["test_acc"])) + 1
    bng = h_ng ["test_acc"].index(max(h_ng ["test_acc"])) + 1
    print(f"  {'最佳 Epoch':<20} {bsg:>10} {bng:>12}")
    print(f"  {'总训练时间 (s)':<20} {h_sgd['time'][-1]:>10.1f} {h_ng['time'][-1]:>12.1f}")

    print("=" * 56)

if __name__ == "__main__":
    main()

  MNIST 手写数字分类：SGD vs NG

[1/3] 下载 & 加载 MNIST 数据集...
  训练集: (60000, 784)  测试集: (10000, 784)

[2/3] 训练两种优化器...

────────────────────────────────────────────────────────
  方法: SGD   lr=0.08   batch=128   epochs=30
────────────────────────────────────────────────────────
  Epoch  1/30 | TrainLoss=0.4930 | TestLoss=0.2968 | TrainAcc=92.30% | TestAcc=91.52% | 0.3s
  Epoch  2/30 | TrainLoss=0.2764 | TestLoss=0.2384 | TrainAcc=94.14% | TestAcc=93.10% | 0.5s
  Epoch  3/30 | TrainLoss=0.2270 | TestLoss=0.2078 | TrainAcc=95.10% | TestAcc=93.98% | 0.7s
  Epoch  4/30 | TrainLoss=0.1952 | TestLoss=0.1797 | TrainAcc=95.98% | TestAcc=94.85% | 0.8s
  Epoch  5/30 | TrainLoss=0.1719 | TestLoss=0.1634 | TrainAcc=96.18% | TestAcc=95.22% | 0.9s
  Epoch  6/30 | TrainLoss=0.1536 | TestLoss=0.1499 | TrainAcc=96.90% | TestAcc=95.79% | 1.0s
  Epoch  7/30 | TrainLoss=0.1389 | TestLoss=0.1403 | TrainAcc=97.10% | TestAcc=95.96% | 1.1s
  Epoch  8/30 | TrainLoss=0.1268 | TestLoss=0.1292 | TrainAcc=97.34% | TestAcc=9